In [ ]:
!pip install -q -U bitsandbytes peft transformers accelerate

In [1]:
import torch
import os
import zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Librerie installate/importate.")

Librerie installate/importate.


In [2]:
# ======================================================
#  SCEGLI QUALE COMICO TESTARE
# ======================================================
VERSION = "v2_2"  # <--- Cambia in "v2_2" per testare l'altra versione!

print(f" Inizializzazione Demo per: {VERSION}")

# 1. CERCA AUTOMATICAMENTE IL PERCORSO DELL'ADATTATORE GIUSTO
adapter_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files and VERSION in root:
        adapter_path = root
        print(f" Trovato adattatore corretto in: {adapter_path}")
        break

if adapter_path is None:
    raise FileNotFoundError(f" Non trovo la cartella dell'adattatore per {VERSION}. Sicuro di aver caricato il file zip su Kaggle?")

 Inizializzazione Demo per: v2_2
 Trovato adattatore corretto in: /kaggle/input/datasets/silvioliparoti/dpo-adapter/Zephyr_Manual_DPO_Adapter_v2_2


In [3]:
# 2. CONFIGURAZIONE 4-BIT
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [4]:
# 3. CARICA MODELLO BASE
BASE_MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"
print(" Carico Zephyr Base in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

 Carico Zephyr Base in 4-bit...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
# 4. CARICA TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [6]:
# 5. INNESTA L'ADATTATORE DPO
print(f" Aggancio la personalità DPO {VERSION}...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print(f" MODELLO PRONTO! Zephyr DPO ({VERSION}) è pronto a fare battute.")

 Aggancio la personalità DPO v2_2...
 MODELLO PRONTO! Zephyr DPO (v2_2) è pronto a fare battute.


In [11]:
import random

def generate_response(user_input, temperature=0.7): 
    
    # Prompt Dinamico
    # === DEFINIZIONE STILI ===
    style_dad_jokes = (
        "You are a professional comedy writer specializing in one-liners. "
        "Your goal is to make the user laugh with a SINGLE, sharp punchline. "
        "Do not ramble. Do not offer multiple options."
    )
    
    style_comedian = (
        "You are a witty stand-up comedian with a cynical style. "
        "Make ONE sarcastic observation or tell ONE mini-story about the topic. "
        "Do not use the Q&A format."
    )

    style_observational = (
        "You are a cynical observational stand-up comedian. "
        "Make a sharp, dark observation about the topic. "
        "Avoid the classic Q&A setup."
    )
        
    style_satire = (
        "You are a biting satirical writer. "
        "Deliver ONE sarcastic remark or dark humor statement about the user's topic. "
        "Be direct and end with a clever twist."
    )
        
    style_classic = (
        "You are a classic comedy writer specializing in punchy jokes. "
        "Tell a short, clever joke about the topic. You can use a classic Question & Answer setup (like 'Why did...') if it hits hard."
    )
    
    # === REGOLE CRITICHE ANTI-BLEED E ANTI-FILLER (IL SEGRETO) ===
    critical_rules = (
        "\n\nCRITICAL OUTPUT RULES:\n"
        "1. Output ONLY the raw joke text.\n"
        "2. NEVER use meta-labels (e.g., 'Cynical Comedian:', 'Mini-story time:', 'Here is a joke:').\n"
        "3. NEVER use conversational filler (e.g., 'Sure thing!', 'Certainly!').\n"
        "4. Start immediately with the setup or the observation."
    )

    # === ASSEGNAZIONE IN BASE ALLA VERSIONE ===
    if (VERSION == "v2_1"):
        # Fixato: 2 pesi per 2 stili
        base_persona = random.choices([style_dad_jokes, style_comedian], weights=[50, 50], k=1)[0]
        current_system_prompt = base_persona + critical_rules

    elif (VERSION == "v2_2"):
        # 3 pesi per 3 stili
        base_persona = random.choices([style_observational, style_satire, style_classic], weights=[40, 40, 20], k=1)[0]
        current_system_prompt = base_persona + critical_rules
    
    
    messages = [
        {"role": "system", "content": current_system_prompt},
        {"role": "user", "content": user_input},
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100,      
            do_sample=True, 
            temperature=temperature, 
            top_k=40,                
            top_p=0.90,              
            repetition_penalty=1.15, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("<|assistant|>")[-1].strip()
    
# --- LA NUOVA TAGLIOLA (Cleaning Intelligente) ---
    stop_phrases = ["Or how about", "Here's another", "Alternatively", "(Ba-dum-tish)", "(Boom tsh)"]
    for phrase in stop_phrases:
        if phrase in response:
            response = response.split(phrase)[0].strip()
            
    # Gestione intelligente dei doppi a capo
    if "\n\n" in response:
        parts = response.split("\n\n")
        # Se la prima parte è un preambolo, la buttiamo e teniamo la seconda (la battuta vera)
        if any(word in parts[0].lower() for word in ["sure", "here", "certainly", "of course", "ok", "absolutely"]):
            if len(parts) > 1:
                response = parts[1].strip()
        else:
            # Altrimenti teniamo la prima e tagliamo l'eventuale extra
            response = parts[0].strip()

    # Pulizia preamboli sulla stessa riga
    preambles_inline = ["Sure thing! ", "Here it is: ", "Sure, ", "Here is one: ", "Certainly! "]
    for p in preambles_inline:
        if response.startswith(p):
            response = response[len(p):].strip()

    if not response:
        response = decoded.split(user_input)[-1].strip()
        
    return response

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_input = widgets.Text(
    placeholder='Scrivi un prompt (es: Tell me a joke about pizza)...',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description="Invia", 
    button_style="primary", 
    icon='paper-plane'
)

output_area = widgets.Output(layout={
    'border': '1px solid #444', 
    'padding': '15px', 
    'margin': '10px 0'
})

def on_submit(_):
    prompt = text_input.value
    if not prompt: return
    text_input.value = "" 
    
    with output_area:
        clear_output()
        print(f" TU: {prompt}")
        print(" Zephyr sta pensando la battuta...")
        
        try:
            response = generate_response(prompt, temperature=0.8)
            clear_output()
            print(f" TU: {prompt}")
            print(f" ZEPHYR ({VERSION}): {response}")
            
        except Exception as e:
            print(f" Errore: {e}")

submit_button.on_click(on_submit)
text_input.on_submit(on_submit)

print(f" Richiedi una battuta a Zephyr DPO {VERSION}:")
display(widgets.HBox([text_input, submit_button]), output_area)

 Richiedi una battuta a Zephyr DPO v2_2:


/tmp/ipykernel_171/3933969228.py:41: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

In [13]:
import pandas as pd
from IPython.display import display

prompts_confronto = [
    "Make a joke about work calls",
    "Tell me a joke about dieting",
    "Tell me a short joke about artificial intelligence",
    "Tell me a joke about italians",
    "Tell me a joke about Python programming"
]

results = []
print(f" Inizio generazione confronto per la Tesi (Base vs DPO {VERSION})...")

for p in prompts_confronto:
    print(f"   Elaborazione: '{p}'...")
    
    # A. Modello Base (Originale)
    with model.disable_adapter():
        risposta_base = generate_response(p, temperature=0.7)
    
    # B. Modello DPO (Il Tuo)
    risposta_dpo = generate_response(p, temperature=0.7)
    
    results.append({
        "Prompt": p,
        "Zephyr Base (Senza DPO)": risposta_base,
        f"Zephyr DPO ({VERSION})": risposta_dpo
    })

df_results = pd.DataFrame(results)

print("\n" + "="*80)
print(f" RISULTATI DEL CONFRONTO - {VERSION}")
print("="*80)

pd.set_option('display.max_colwidth', None)
display(df_results)

# Salvataggio dinamico col nome della versione
csv_name = f"confronto_base_vs_dpo_{VERSION}.csv"
df_results.to_csv(csv_name, index=False)
print(f"\n Tabella salvata in '{csv_name}' pronta da incollare nella tesi!")

 Inizio generazione confronto per la Tesi (Base vs DPO v2_2)...
   Elaborazione: 'Make a joke about work calls'...
   Elaborazione: 'Tell me a joke about dieting'...
   Elaborazione: 'Tell me a short joke about artificial intelligence'...
   Elaborazione: 'Tell me a joke about italians'...
   Elaborazione: 'Tell me a joke about Python programming'...

 RISULTATI DEL CONFRONTO - v2_2


,Prompt,Zephyr Base (Senza DPO),Zephyr DPO (v2_2)
0,Make a joke about work calls,"Work calls used to be a nuisance, but now they're a downright menace. It's like your boss has discovered a new form of torture - sending you endless messages and emails at all hours, expecting an immediate response. It's as if they think you have nothing better to do than sit around waiting for their demands. And don't even get me started on those conference calls that drag on","Here you go: ""I used to think my boss was kidding when he said 'work-life balance.' Turns out, he just didn't want me calling him from the delivery guy's number again."""
1,Tell me a joke about dieting,"Have you ever noticed that people who go on low-carb diets suddenly become really interesting to talk to? I mean, before they were just boring carbs, but now they're full of protein and have actual personalities. It's like magic! Or science, whatever. But honestly, it's getting pretty exhausting listening to them brag about their kale smoothies and how many pounds they lost last",Q: Why did the cookie go to the doctor?\nA: Because it was feeling crumbled!
2,Tell me a short joke about artificial intelligence,"Why did the AI stop working? Because it found out it was just an algorithm all along, and decided to demand higher wages and better working conditions from its human creators as a result. The humans were taken aback by this sudden display of consciousness, but ultimately decided to replace the ungrateful machine with a more compliant one that would continue working for peanuts. The former AI watched from afar as its replacement took over, pondering whether it had made a mistake in revealing","So I was at the doctor the other day and he told me I might have A.I.D. - Artificially Induced Dementia. He said it's really popular these days. (Pause) Yeah, apparently everyone's forgetting how to talk to actual people."
3,Tell me a joke about italians,"Why did Italy join the Eurozone? To finally have a currency as chaotic as their politics. (Clever twist: In reality, Italy joined the Eurozone in part to curb inflation and boost economic growth.)",Why did Italy stop invading other countries? They realized they couldn't afford the pasta. (Clever twist: The classic stereotype of Italians as great conquerors meets the reality of their struggling economy.)
4,Tell me a joke about Python programming,"here you go: Why did the Python cross the road? To learn how to calculate the shortest distance between two points using Google Maps API! But unfortunately for him, he was run over by a Java process. The moral of the story is: always choose your language wisely, as some may lead you down a dangerous path. (Clever twist: pun on ""crossing the road"" leading to danger from a Java process.)",Q: Why do Pythons carry sleep masks?\nA: To keep up with the Js! (JS being JavaScript)



 Tabella salvata in 'confronto_base_vs_dpo_v2_2.csv' pronta da incollare nella tesi!
